#PYSPARK

In [ ]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("PySpark Operations Demo") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [ ]:
import pandas as pd

sales_raw = {
    'transaction_id': [1, 2, 3, 4, 5, 5, 6, 7],
    'product_id': [101, 102, 101, 103, 102, 102, 104, 101],
    'amount': [150.0, 300.0, 150.0, 450.0, 300.0, 300.0, 100.0, 200.0],
    'region': ['North', 'South', 'North', 'East', 'South', 'South', 'West', 'East']
}
pd.DataFrame(sales_raw).to_csv('sales_data.csv', index=False)

products_raw = {
    'product_id': [101, 102, 103, 104],
    'product_name': ['Laptop', 'Smartphone', 'Tablet', 'Headphones'],
    'category': ['Electronics', 'Electronics', 'Electronics', 'Accessories']
}
pd.DataFrame(products_raw).to_csv('products_data.csv', index=False)

print("Mock CSV files 'sales_data.csv' and 'products_data.csv' successfully created!")

Mock CSV files 'sales_data.csv' and 'products_data.csv' successfully created!


In [ ]:
sales_df = spark.read.csv('sales_data.csv', header=True, inferSchema=True)

sales_df.printSchema()

sales_df.show()

root
 |-- transaction_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)

+--------------+----------+------+------+
|transaction_id|product_id|amount|region|
+--------------+----------+------+------+
|             1|       101| 150.0| North|
|             2|       102| 300.0| South|
|             3|       101| 150.0| North|
|             4|       103| 450.0|  East|
|             5|       102| 300.0| South|
|             5|       102| 300.0| South|
|             6|       104| 100.0|  West|
|             7|       101| 200.0|  East|
+--------------+----------+------+------+



In [ ]:
deduplicated_sales_df = sales_df.dropDuplicates()

print("Original Count:", sales_df.count())
print("Deduplicated Count:", deduplicated_sales_df.count())
deduplicated_sales_df.show()

Original Count: 8
Deduplicated Count: 7
+--------------+----------+------+------+
|transaction_id|product_id|amount|region|
+--------------+----------+------+------+
|             4|       103| 450.0|  East|
|             6|       104| 100.0|  West|
|             1|       101| 150.0| North|
|             7|       101| 200.0|  East|
|             3|       101| 150.0| North|
|             2|       102| 300.0| South|
|             5|       102| 300.0| South|
+--------------+----------+------+------+



In [ ]:
filtered_agg_df = deduplicated_sales_df \
    .filter(col("amount") > 100) \
    .groupBy("region") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_sales")

filtered_agg_df.show()

+------+-----------+
|region|total_sales|
+------+-----------+
| South|      600.0|
|  East|      650.0|
| North|      300.0|
+------+-----------+



In [ ]:
products_df = spark.read.csv('products_data.csv', header=True, inferSchema=True)

joined_df = deduplicated_sales_df.join(products_df, on="product_id", how="inner")

joined_df.show()

+----------+--------------+------+------+------------+-----------+
|product_id|transaction_id|amount|region|product_name|   category|
+----------+--------------+------+------+------------+-----------+
|       103|             4| 450.0|  East|      Tablet|Electronics|
|       104|             6| 100.0|  West|  Headphones|Accessories|
|       101|             1| 150.0| North|      Laptop|Electronics|
|       101|             7| 200.0|  East|      Laptop|Electronics|
|       101|             3| 150.0| North|      Laptop|Electronics|
|       102|             2| 300.0| South|  Smartphone|Electronics|
|       102|             5| 300.0| South|  Smartphone|Electronics|
+----------+--------------+------+------+------------+-----------+



In [ ]:
avg_sales_by_category = joined_df \
    .groupBy("category") \
    .agg(avg("amount").alias("average_sales"))

avg_sales_by_category.show()

+-----------+-----------------+
|   category|    average_sales|
+-----------+-----------------+
|Electronics|258.3333333333333|
|Accessories|            100.0|
+-----------+-----------------+

